In [1]:
import pandas as pd
import numpy as np
import string
import re

from rapidfuzz import process, fuzz, distance


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
cols_to_use = ['filer_committee_id_number',
                    'form_type',
                    'payee_organization_name',
                    'payee_last_name',
                    'payee_first_name',
                    'expenditure_date',
                    'expenditure_amount',
                    'expenditure_purpose_descrip',
                    'beneficiary_candidate_last_name',
                    'beneficiary_candidate_first_name',
                    'memo_code',
                    'memo_text_description']

# Load data 
sb_data = pd.read_csv('../data/raw/SB_dataset.csv', usecols = cols_to_use)


/var/folders/cl/bdrh3m8j0fdfj6zm7zhhc2ch0000gn/T/ipykernel_76724/1335903702.py:15: DtypeWarning: Columns (7,8,20,27,28,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  sb_data = pd.read_csv('../data/raw/SB_dataset.csv', usecols = cols_to_use)


## Process SB Data

We start by dropping any rows with missing filer committee ID numbers and any rows in which the filer committee ID number does not match the C####### pattern we expect

In [3]:
# drop rows where filer_committee_id_number is null
sb_data = sb_data.dropna(subset=['filer_committee_id_number'])

# drop rows without a properly formated committee id
pattern = r'^C\d{8}$'
sb_data = sb_data[sb_data['filer_committee_id_number'].str.contains(pattern, na=False)]


### Form Types
Form types code each expenditure into categories defined by the FEC. All form types in the SB (schedule B - expenditure) data should begin with the letters SB. 

Looking at the data there is 1 observation where this is not the case. It is associated with payments to ActBlue for credit card processing expenses. Given the small sample, I will update the form type to reflect SB21.  

SB17 isn't a disbursement lineitem associated with PAC filings. There is one observation with this coding from the 2023 Year End filing (1761077) from the Ryman Hospitality Properties Political Action Committee (C00183707). I'm going to reassign this to SB23, as it is described as a campaign contribution.

SB27 isn't a disbursement lineitem associated with PAC filings. I'm going to reassign this to SB29, as it is described as a mistaken loan.

| Form Type       | Description         |
|-----------------|---------------------|
| SB21            | Operating Expenditures |
| SB21B           | Other Operating Expenditures |
| SB22            | Transfers to Affiliated  Other Party Committees |
| SB23            | Contributions to Fed. Candidates Committees and other political committees |
| SB26            | Loan repayments made |
| SB28A           | Refunds to Individuals |
| SB28B           | Refunds to Party Committees |
| SB28C           | Refunds to Other Committees |
| SB29            | Other Disbursements |
| SB30B           | Federal Election Activity |



In [4]:
# Recode obviously miscodded form types
sb_data.loc[sb_data['form_type'] == 'SB17', 'form_type'] = 'SB23'
sb_data.loc[sb_data['form_type'] == '1b', 'form_type'] = 'SB21B'
sb_data.loc[sb_data['form_type'].isin(['27','SB27']), 'form_type'] = 'SB29'

# Capitalize form_type to standardize SB21b SB21B etc
sb_data['form_type'] = sb_data['form_type'].str.upper()

In [5]:
# Create form_type_desc column to help me remember what i'm looking at 
form_type_map = {
    'SB21' : 'Operating Expenditures',
    'SB21B' : 'Other Operating Expenditures',
    'SB22' : 'Transfers to Affiliated Committees',
    'SB23' : 'Contributions to Candidates and Committees', 
    'SB26' : 'Loan repayments made',
    'SB28A' : 'Refunds to Individuals',
    'SB28B' : 'Refunds to Party Committees',
    'SB28C' : 'Refunds to Other Committees',
    'SB29' : 'Other Disbursements',
    'SB30B': 'Federal Election Activity',
}

sb_data['form_type_desc'] = sb_data['form_type'].map(form_type_map)


In [6]:
sb_data.form_type_desc.value_counts()

form_type_desc
Contributions to Candidates and Committees    580591
Other Operating Expenditures                  417375
Other Disbursements                           152392
Refunds to Individuals                        116754
Transfers to Affiliated Committees             15473
Refunds to Other Committees                      335
Loan repayments made                             150
Federal Election Activity                        147
Refunds to Party Committees                       68
Operating Expenditures                            19
Name: count, dtype: int64

For the purpose of this work, we are not interested in refunds so we'd like to drop all refund rows. First we must check to make sure there isnt evidence of large scale mis categorization of expenses as refunds.

In [7]:
# Check to make sure everything in refunds categories contain refund (or seomthing related) as purpose descrips
print("Refunds:")
sb_data[(sb_data['form_type'].isin(['SB28A','SB28B''SB28C'])) & 
        ~sb_data['expenditure_purpose_descrip'].str.contains('REFUND|VOID|REIMBURSE|ERRO|RETURN', na = False, case = False) ].expenditure_purpose_descrip.unique().tolist()


Refunds:


[nan,
 'Duplicate PayPal Transaction',
 'SEE MEMO ATTACHED',
 'Reissue of 12/13/2022 check',
 'Contribution to Committee',
 'Contribution Ref to Corporation',
 'Contribution Ref to Individual',
 'See Line 28a - US Travel Association',
 'GiveSmart',
 'Mark C Mckean ran card by mistake',
 'Contribution Rerfund',
 'Dispute Reversed',
 'Original contribution 3/3/24',
 'Earmarked through ACTBlue',
 'See Line 28 - US Travel Association',
 'See Line 28, U.S. Travel Association',
 'Chargeback',
 'Donor made Chargeback',
 'Member accidently put $1.00 to the PAC fund.  Member corrected PAC contribution to 0.05 cents.',
 'Adjustment to Payroll Deduction',
 'ANTONIA SACCHETTI',
 'Disputed contribution, originally received 4/21/23',
 'Fundraising Consulting Fee',
 'Legal Fees',
 'PARKING SERVICES',
 'Mistaken contribution',
 'Donor incorrectly selected monthly recurring payment.',
 'Meeting venue',
 'Print cards for Holiday Party',
 'Flowers for retiring Club President',
 'Repay Contribution Made o

In [8]:
# filter out refunds 
spending_data = sb_data[~sb_data['form_type_desc'].str.contains('Refund', na=False)].copy()

In [9]:
spending_data.form_type_desc.value_counts()

form_type_desc
Contributions to Candidates and Committees    580591
Other Operating Expenditures                  417375
Other Disbursements                           152392
Transfers to Affiliated Committees             15473
Loan repayments made                             150
Federal Election Activity                        147
Operating Expenditures                            19
Name: count, dtype: int64

In [10]:
# A lot of the other operating expenditures are contributions to committees. I want to recategorize those to contributions to committees because I'll try to clean those differently 

cont_cmte_pattern = 'FOR (?:CONGRESS|SENATE|STATE|ASSEMB|GOV|ALABAMA|\bAL\b|ALASKA|\bAK\b|ARIZONA|\bAZ\b|ARKANSAS|\bAR\b|CALIFORNIA|\bCA\b|COLORADO|\bCO\b|CONNECTICUT|\bCT\b|DELAWARE|\bDE\b|FLORIDA|\bFL\b|GEORGIA|\bGA\b|HAWAII|\bHI\b|IDAHO|\bID\b|ILLINOIS|\bIL\b|INDIANA|\bIN\b|IOWA|\bIA\b|KANSAS|\bKS\b|KENTUCKY|\bKY\b|LOUISIANA|\bLA\b|MAINE|\bME\b|MARYLAND|\bMD\b|MASSACHUSETTS|\bMA\b|MICHIGAN|\bMI\b|MINNESOTA|\bMN\b|MISSISSIPPI|\bMS\b|MISSOURI|\bMO\b|MONTANA|\bMT\b|NEBRASKA|\bNE\b|NEVADA|\bNV\b|NEW HAMPSHIRE|\bNH\b|NEW JERSEY|\bNJ\b|NEW MEXICO|\bNM\b|NEW YORK|\bNY\b|NORTH CAROLINA|\bNC\b|NORTH DAKOTA|\bND\b|OHIO|\bOH\b|OKLAHOMA|\bOK\b|OREGON|\bOR\b|PENNSYLVANIA|\bPA\b|RHODE ISLAND|\bRI\b|SOUTH CAROLINA|\bSC\b|SOUTH DAKOTA|\bSD\b|TENNESSEE|\bTN\b|TEXAS|\bTX\b|UTAH|\bUT\b|VERMONT|\bVT\b|VIRGINIA|\bVA\b|WASHINGTON|\bWA\b|WEST VIRGINIA|\bWV\b|WISCONSIN|\bWI\b|WYOMING|\bWY\b)|ELECT\b|FRIENDS OF|\bPAC\b?|\bCOMMITTEE\b?|\bCMTE\b|\bVICTORY\b|\bFUND\b?'

recat_payee_org_name = spending_data[(spending_data['form_type_desc'].isin(['Other Operating Expenditures','Other Disbursements'])) & (spending_data['payee_organization_name'].str.contains(pat = cont_cmte_pattern, regex = True, case = False) ) ].payee_organization_name.unique()

# Check to make sure this is working like I think it should
recat_payee_org_name


array(['Friends of Kim Ward', 'LISA MCCLAIN FOR CONGRESS',
       'Amy Walen for State House', ..., 'Chas Cannon for State House',
       'Meeks for Georgia', 'Cmte to Elect Patty Stinson for State House'],
      dtype=object)

In [11]:
# Recode the form_type 
spending_data['form_type'] = np.where(
                                (spending_data['payee_organization_name'].isin(recat_payee_org_name)) & (spending_data['form_type_desc']=='Other Operating Expenditures'),
                                'SB23',
                                spending_data['form_type']
)

# Recode the form_type_desc
spending_data['form_type_desc'] = np.where(
                                (spending_data['payee_organization_name'].isin(recat_payee_org_name)) & (spending_data['form_type_desc']=='Other Operating Expenditures'),
                                'Contributions to Candidates and Committees',
                                spending_data['form_type_desc']
)

spending_data.form_type_desc.value_counts()

form_type_desc
Contributions to Candidates and Committees    581186
Other Operating Expenditures                  416780
Other Disbursements                           152392
Transfers to Affiliated Committees             15473
Loan repayments made                             150
Federal Election Activity                        147
Operating Expenditures                            19
Name: count, dtype: int64

In [12]:
# Define numeric columns as numeric
spending_data['expenditure_amount'] = pd.to_numeric(spending_data['expenditure_amount'])


In [13]:
# Normalize all string data 

def normalize_string_columns(df):
    """
    Function to normalize all string columns to uppercase, remove punctuation, strip whitespace 
    """
    for col in df.columns:
        # If the column is an object
        if df[col].dtype == 'object':
           df[col] = (
           df[col]
           # remove punctuation
           .str.replace(f"[{string.punctuation}]", " ", regex=True)
           # remove trailing whitespace and multiple spaces
           .str.replace(r"\s+"," ", regex = True)
           # remove common suffixes
           .str.replace(r'\b(LLC|INC|LTD|LLP|PLCC|PLC|PC|STRATEGIES|SERVICES|ASSOCIATES|CONSULTING|GROUP|CORPORATION|COM|ORG|PAC|PARTNERS|ASSOCIATION|NET|CONTRIBUTIONS)\b$', '', regex=True, case=False)
           # remove numbers by themselves ie 1223 but not 1223abc
           .str.replace(r'\b\d+\b\.?', '', regex=True, case=False)
           # remove whitespaces again
           .str.strip()
           # capitalize everything
           .str.upper()
           )
    return df

# Apply to our DF
spending_data = normalize_string_columns(spending_data) 
 

In [14]:
## columns are filled out depending on the payee type - an organization, individual, or candidate, I want one column that concatenates all of this information

# Build component strings safely, ensuring no NaN text appears
spending_data['payee_name'] = (
    spending_data['payee_last_name'].fillna('') + ' ' +
    spending_data['payee_first_name'].fillna('')
    # removed city because we were grouping all of the payees from the same city together 
).str.strip()

spending_data['beneficiary_cand_name'] = (
    spending_data['beneficiary_candidate_first_name'].fillna('') + ' ' +
    spending_data['beneficiary_candidate_last_name'].fillna('')
).str.strip()

# Select the first non-null (and non-empty) among the three
spending_data['payee_id'] = (
    spending_data[['payee_organization_name', 'payee_name', 'beneficiary_cand_name']]
    .replace('', pd.NA)         
    .bfill(axis=1)              
    .iloc[:, 0]                 
)

n_original_payees = len(spending_data['payee_id'].unique())



In [15]:
# drop rows where payee_id is null 
spending_data.dropna(subset='payee_id', inplace=True)
spending_data.isna().sum()

form_type                                 0
filer_committee_id_number                 0
payee_organization_name               68124
payee_last_name                     1097884
payee_first_name                    1097887
expenditure_date                          0
expenditure_amount                        0
expenditure_purpose_descrip           40562
beneficiary_candidate_last_name      793925
beneficiary_candidate_first_name     794095
memo_code                            811658
memo_text_description                850078
form_type_desc                            0
payee_name                                0
beneficiary_cand_name                     0
payee_id                                  0
dtype: int64

In [16]:
# Replace state abbreviations in strings with state names to futher standardize texts
# EX. TEXAS DEMOCRATIC PARTY -> TX DEMOCRATIC PARTY
state_map = {
    'ALABAMA': 'AL',
    'ALASKA': 'AK',
    'ARIZONA': 'AZ',
    'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA',
    'COLORADO': 'CO',
    'CONNECTICUT': 'CT',
    'DELAWARE': 'DE',
    'FLORIDA': 'FL',
    'GEORGIA': 'GA',
    'HAWAII': 'HI',
    'IDAHO': 'ID',
    'ILLINOIS': 'IL',
    'INDIANA': 'IN',
    'IOWA': 'IA',
    'KANSAS': 'KS',
    'KENTUCKY': 'KY',
    'LOUISIANA': 'LA',
    'MAINE': 'ME',
    'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA',
    'MICHIGAN': 'MI',
    'MINNESOTA': 'MN',
    'MISSISSIPPI': 'MS',
    'MISSOURI': 'MO',
    'MONTANA': 'MT',
    'NEBRASKA': 'NE',
    'NEVADA': 'NV',
    'NEW HAMPSHIRE': 'NH',
    'NEW JERSEY': 'NJ',
    'NEW MEXICO': 'NM',
    'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC',
    'NORTH DAKOTA': 'ND',
    'OHIO': 'OH',
    'OKLAHOMA': 'OK',
    'OREGON': 'OR',
    'PENNSYLVANIA': 'PA',
    'RHODE ISLAND': 'RI',
    'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD',
    'TENNESSEE': 'TN',
    'TEXAS': 'TX',
    'UTAH': 'UT',
    'VERMONT': 'VT',
    'VIRGINIA': 'VA',
    'WASHINGTON': 'WA',
    'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI',
    'WYOMING': 'WY'
}


pattern = r'\b(' + '|'.join(state_map.keys()) + r')\b'

def replace_state_abbrev(string_):
    """
    Use state_map to replace all state names with state abbrev
    to further normalize strings
    """
    return re.sub(pattern,
                  lambda x: state_map[x.group(0)],
                  string_)

# Apply replace state abbrev to payee_id 
spending_data['payee_id'] =  spending_data['payee_id'].apply(replace_state_abbrev)

In [17]:
# Retry the suffix stripping in cases where there may have been multiple 
spending_data['payee_id'] = spending_data['payee_id'].str.replace(r'\s*(INC|CPA|COMPANY|CO|N A|LLC|P C)\s*$','',regex = True, case = False)

spending_data[spending_data['form_type']=='SB21B'].payee_id.unique().tolist()

['KEVIN L BOYCE COMMITTEE',
 'CFS COMPLIANCE',
 'THE TOWNSEND',
 'TRUIST',
 'COMERICA BANK',
 'BIPARTISAN CLIMATE ACTION',
 'JPMORGAN CHASE BANK',
 'SQUARESPACE',
 'ARISTOTLE INTERNATIONAL',
 'FACEBOOK',
 'GOOGLE',
 'HAWATMEH NICK',
 'PROFESSIONAL DATA',
 'ACTBLUE',
 'AMALGAMATED BANK',
 'KESWICK HALL',
 'TIBER CREEK',
 'AMERICAN EXPRESS',
 'LYFT RIDE',
 'UBER',
 'HYATT REGENCY ORLANDO',
 'MARCO PROMOTIONAL PRODUCTS',
 'JOSELITO CASA',
 'CUSTOM INK',
 'L ARDENTE',
 'UNITED AIRLINES',
 'AMERICAN AIRLINES',
 'MLT STRATEGIC FUNDRAISING',
 'PENDRY MANHATTAN',
 'AMTRAK',
 'HOTEL SAN FRANCIS',
 'ALL ABOUT PARKINSAN',
 'DONNELL PAYNE',
 'BROWNS CAB',
 'FARAWAY NANTUCKET',
 'HILTON GARDEN INN',
 'NGP VAN',
 'TOTAL WINE MORE',
 'AT ANY TABLE',
 'SAVOYA',
 'PURPURA SALVATORE',
 'HUCKABY DAVIS LISKER',
 'WINRED',
 'NASW',
 'LIEB SARAH',
 'CREST STRATEGIES',
 'WELLS FARGO',
 'THE MONEY WHEEL',
 'EVERYACTION',
 'AMALGAMATED',
 'VICTORY FORWARD SOLUTIONS',
 'THE COLIBRI COLLECTIVE',
 'CHAIN BRIDGE B

In [18]:
# Look at the largest vendors by amount paid by pacs
spending_data[spending_data['form_type']=='SB21B'].groupby('payee_organization_name').expenditure_amount.sum().sort_values(ascending=False)


payee_organization_name
MOTHERSHIP                                                                                                  6.496067e+07
LAUNCHPAD STRATEGIES                                                                                        5.863227e+07
UNITED STATES OF AMERICA                                                                                    5.200895e+07
WINRED TECHNICAL SERVICES                                                                                   5.194731e+07
COINBASE                                                                                                    4.801604e+07
META PLATFORMS INC                                                                                          4.675449e+07
AMERICAN EXPRESS                                                                                            4.319817e+07
TARGETED VICTORY                                                                                            3.738768e+07
EVENT ST

In [19]:
# Define dictionary for large vendors
#https://medium.com/@_bryceli/using-dictionaries-as-regex-in-python-de9033bb3e0f

def parse(input_str):
    # Dictionary with regex keys will be used as the pattern by turning it into a list then to a string.
    vendor_name_dict = {
        #'regex pattern': 'category'
        r'ACTBLUE|ACT BLUE|ACTBLEU|ACT BLEU|ACBLUE': 'ActBlue',
        r'WINRED|WIN RED': 'WinRed',
        r'\s?HILTON\s?|CONRAD (?:NY|NEW|WASHINGTON)|EMBASSY SUITES?|HAMPTON INN|HOME2|HOME\s?WOOD|WALDORF': 'HILTON HOTEL',
        r'UNITED STATES POSTAL|USPS|U S POSTAL': 'USPS',
        r'AMERIC?AN EXPRESS|AM\s?EX': 'AMERICAN EXPRESS',
        r'NGP\s?VAN|EVERY\s?ACTION': 'NGPVAN',
        r'FACEBOOK|META|INTEGRAM|INSTAGRAM': 'META',
        r'SIEU': 'SIEU',
        r'7 ELEVEN': '7ELEVEN',
        r'AC HOTEL|MARRIOTT|ALOFT|CITIZENM|RITZ CARLTON|ST REGIS|W HOTEL|SHERATON|DELTA HOTEL|WESTIN|RENAISSANCE|COURT\s?YARD HOTEL|SPRINGHILL SUITES?|DOUBLE\s?TREE|ELEMENT\b|FAIRFIELD\b|FOUR POINTS|\bJW\b|MERIDIEN|MOXY|RESIDENCE INN':'MARRIOTT HOTEL',
        r'\b(?:ACEC|AMERICAN COUNCIL OF ENGINEERING)\b':'ACEC PAC',
        r'\b(?:ACLI|AMERICAN COUNCIL OF LIFE INS)':'ACLI PAC',
        r'\bADOBE\b':'ADOBE',
        r'\bADP\b':'ADP',
        r'\bADT\b':'ADT',
        r'\bAFL\s?CIO\b': 'AFL CIO',
        r'ALAMO.+CAR': 'ALAMO CAR RENTAL',
        r'ALASKAN? AIR':'ALASKA AIRLINES',
        r'\bALLIANZ':'ALLIANZ TRAVEL SERVICES',
        r'AMALGAMATED': 'AMALGAMATED BANK',
        r'\b(?:AMAZON|AWS)\b': 'AMAZON',
        r'AMERICAN AIR': 'AMERICAN AIRLINES',
        r'AMERICAN LEGION':'AMERICAN LEGION',
        r'AMERICAN PROPERTY CASUALTY|APCIA':'APCIA PAC',
        r'ANNE LEWIS|MISSIONWIRED':'MISSIONWIRED',
        r'APPLE (?:CO\w+|ONLINE)': 'APPLE',
        r'ARIZONA DEMOCRATIC PARTY':'ARIZONA DEMOCRATIC PARTY',
        r'ASSEMBLY DEMOCRATIC CAMP|ASSEMBLY DEMOCRATIC COMM':'ASSEMBLE DEMOCRATIC CAMPAIGN COMMITEE',
        r'\bAT T\b': 'AT&T',
        r'\bLOCAL \d+\b|^LOCAL (?:\d+|(?:A )?INTERNATIONAL|IUOE|UNITE|TEAMSTER|SCHOOL|UNION)':'LOCAL UNION',
        r'\bAVIS\b':'AVIS RENTAL CAR',
        r'BANK OF AMERICA':'BANK OF AMERICA',
        r'\bBB T\b':'BB&T BANK',
        r'BELLAGIO':'BELLAGIO HOTEL',
        r'\bBEST\s?BUY\b': 'BEST BUY',
        r'\bBEST WESTERN\b': 'BEST WESTERN HOTEL',
        r'BLOOMBERG':'BLOOMBERG',
        r'BLUE\s?CROSS BLUE|BCBS|BLUE\s?PAC':'BLUE CROSS BLUE SHIELD',
        r'\bBMO\b':'BMO BANK',
        r'BOLD ACTIVE CON':'BOLD ACTIVE CONSERVATIVES PAC',
        r'BOSTON GLOBE':'BOSTON GLOBE',
        r'BOYS AND GIRLS CLUB|BOYS GIRLS CLUB':'BOYS AND GIRLS CLUB',
        r'BRETTPAC': 'BRETTPAC',
        r'BUDGET R':'BUDGET RENT A CAR',
        r'\bBWI\b':'BWI AIRPORT',
        r'CA LUV PAC': 'CA LUV PAC',
        r'\bCANVA\b':'CANVA',
        r'CAPITAL BANK':'CAPITAL BANK',
        r'CAPITAL GRILL':'CAPITAL GRILLE',
        r'CAPTIAL ONE [^ARENA]': 'CAPITAL ONE',
        r'CARMINES?': 'CARMINES',
        r'\bCAVA\b': 'CAVA',
        r'CHAIN\s?BRIDGE': 'CHANI BRIDGE BANK',
        r'CHARLIE PALMER STEAK':'CHARLIE PALMER STEAK HOUSE',
        r'CHARTER COMMUNICATIONS': 'CHARTER COMMUNICATIONS PAC',
        r'^CHASE (?:CARD|BANK|CO|CREDIT|VISA)': 'CHASE BANK',
        r'CHESTER CO\w+ DEM':'CHESTER COUNTY DEMOCRATIC COMMITEE',
        r'CHEVRON':'CHEVRON',
        r'CHICK? FIL A': 'CHICK FIL A',
        r'CHILI S\b': 'CHILIS',
        r'CHIPOTLE': 'CHIPOTLE',
        r'CHOICE HOTELS?|COMFORT INN|COUNTRY INN': 'CHOICE HOTELS',
        r'CIBC\b': 'CIBC BANK',
        r'CIRCLE K\b':'CIRCLE K',
        r'CITGO':'CITGO',
        r'\bCITI\s?(?:BANK|CARDS?|VISA|BUSINESS|CREDIT)':'CITI BANK',
        r'CITIZENS BANK':'CITIZEN BANK',
        r'COLONIAL PARK':'COLONIAL PARKING',
        r'COMCAST':'COMCAST',
        r'CONGRESSIONAL BLACK|\bCBC\b':'CONGRESSIONAL BLACK CAUCUS',
        r'CONGRESSIONAL HISPANIC|\bCHC\b':'CONGRESSIONAL HISPANIC CAUCUS',
        r'COSTCO\b':'COSTCO',
        r'CROWN PLAZA|HOLIDAY INN|IHG HOTEL|INTERCONTINENTAL|KIMP?TON|YOURS TRULY':'IHG HOTEL AND RESORTS',
        r'CURB (?:w\+)? TAXI': 'CURB TAXI',
        r'\bCVS\b':'CVS',
        r'\bCWA\b':'CWA UNION',
        r'DATA FOR PROGRESS|DATA FOR THE SOCIAL GOOD':'DATA FOR PROGRESS',
        r'\bDCA\b':'DCA AIRPORT',
        r'\bDCCC\b|DEMOCRATIC CONGRESSIONAL CAMP':'DCCC',
        r'DEL\s?FRISCO':'DEL FRISCOS',
        r'DELAWARE COUNTY DEM':'DELAWARE COUNTY DEMOCRATS',
        r'^DELTA$|DELTA AIR':'DELTA AIRLINES',
        r'DELUX (?:BUSINESS|CHECKS|ENTERPRISE|FOR|PRINTING)':'DELUX BUSINESS',
        r'DEMOCRATIC ATTORNEYS? GENERALS?':'DEMOCRATIC ATTORNEYS GENERAL ASSOCIATION',
        r'DEMOCRATIC GOVERNORS ASSOC|\bDGA\b':'DEMOCRATIC GOVERNORS ASSOCIATION',
        r'\bDLCC\b|DEMOCRATIC LEGISLATIVE CAMP':'DLCC',
        r'DEMOCRATIC (?:LIEUTENANT|LT) GOVERNORS?|\bDLGA\b':'DEMOCRATIC LIEUTENANT GOVERNORS ASSOCIATION',
        r'DEMOCRATIC MAJORITY':'DEMOCRATIC MAJORITY',
        r'\bDNC\b|DEMOCRATIC NATIONAL COM':'DNC',
        r'\bDSCC\b|DEMOCRATIC (?:SENATE|SENATORIAL) CAM':'DSCC',
        r'DEPARTMENT OF (?:THE |THE US )?TREASUR|\bIRS\b|INTERNAL REVENUE SERVICE': 'IRS',
        r'DISCOVER\s?(?:CARD|CREDIT)':'DISCOVER CARD',
        r'DISNEY':'DISNEY',
        r'DOLLAR A DAY|DOLLAR RENT':'DOLLAR RENT A CAR',
        r'DOMINION ENERGY':'DOMINION ENERGY',
        r'DOMINO\s?S':'DOMINOS PIZZA',
        r'DOOR\s?DASH':'DOOR DASH',
        r'DROP\s?BOX':'DROP BOX',
        r'\bDTW\b':'DTW AIRPORT',
        r'DUNKIN(?: DONUTS)?':'DUNKIN DONUTS',
        r'EINSTEIN (?:BROS|BROTHERS|BAGELS)':'EINSTEIN BROS BAGELS',
        r'\bEMERGE\b':'EMERGE ORG',
        r'EMILY\s?S LIST':'EMILYS LIST',
        r'EMPLOYERS? (?:\w+)? INS':'EMPLOYERS INSURANCE',
        r'\bENCORE\b|WYNN LAS':'ENCORE RESORTS',
        r'ENTERPRISE (?:CAR|HOLDING|R A C|RAC|RENT|TOLL)':'ENTERPRISE RAC',
        r'EQUALITY CALIFORNIA':'EQUALITY CALIFORNIA',
        r'EVERYTOWN FOR':'EVERYTOWN FOR GUN SAFETY',
        r'EXPEDIA|EXPIDA':'EXPEDIA',
        r'EXXON':'EXXON',
        r'EZ\s?CATER':'EX CATER',
        r'E\s?Z\s?PASS':'EZ PASS',
        r'FAIRMONT\b':'FAIRMONT HOTELS',
        r'\bFEC\b|FEDERAL ELECTIONS? COMM':'FEDERAL ELECTIONS COMMISSION',
        r'FED\s?EX|FEDERAL EXPRESS':'FEDEX',
        r'FIDELITY':'FIDELITY',
        r'FIRST (?:CITIZENS |NATIONAL |REPUBLIC )?BANK|FNB':'FIRST BANK',
        r'FONTAINBLE':'FONTAINBLEU',
        r'FORWARD MAJORITY':'FORWARD MAJORITY PAC',
        r'FOUNDING FARMERS?':'FOUNDING FARMERS',
        r'FOUR SEASONS?':'FOUR SEASONS HOTEL',
        r'FRESH MARKET':'FRESH MARKET',
        r'FRONTIER':'FRONTIER AIRLINES',
        r'FUTURE FORWARD':'FUTURE FORWARD PAC',
        r'GET\s?THRU':'GET THRU',
        r'GIANT':'GIANT SUPERMARKET',
        r'GIRL SCOUT':'GIRL SCOUTS',
        r'GO\s?DADDY':'GO DADDY',
        r'GOOGLE|GSUITE':'GOOGLE',
        r'GRAND BOH':'GRAND BOHEMIAN',
        r'\bHYATT\b':'HYATT HOTEL',
        r'GRASSROOTS DEM':'GRASSROOTS DEMOCRATS',
        r'GROUND\s?GAME':'GROUND GAME',
        r'NOMINEE FUND':'NOMINEE FUND',
        r'GRUB\s?HUB':'GRUBHUB',
        r'^GULF$|GULF OIL':'GULF',
        r'\bGUSTO\b':'GUSTO',
        r'HAPPY [C|D]OG':'HAPPY DOG',
        r'HARD ROCK':'HARD ROCK HOTEL',
        r'HARRIS\s?TEETER':'HARRIS TEETER',
        r'(?:BIDEN|HARRIS) VICTORY?|(?:BIDEN|HARRIS) FOR':'BIDEN/HARRIS COMMITTEES',
        r'HART RESEARCH':'HART RESEARCH',
        r'HAWKEYE':'HAWKEYE PAC',
        r'HEARTLAND (?:PAYROLL|PAYMENT|TRAVEL)':'HEARTLAND PAYROLL',
        r'HERTZ(?: CAR| AUTO| FINANCIAL| GLOBAL| HQ| RENT| SYSTEMS)|HER?TZ$':'HERTZ CAR RENTAL',
        r'^HOTELS?\b|^INN AT':'HOTEL',
        r'HOUSE MAJORITY':'HOUSE MAJORITY PAC',
        r'HOUSE REP\w+ CA|\bHRCC\b': 'HOUSE REPUBLICAN CAMPAIGN COMMITTEE',
        r'HOUSE DEM\w+ CA|\bHDCC\b': 'HOUSE DEMOCRATIC CAMPAIGN COMMITTEE',
        r'HOUSE REPw\+ MAJORITY':'HOUSE REPUBLICAN MAJORITY PAC',
        r'HOUSE VICTORY':'HOUSE VICTORY PAC',
        r'HSB?C':'HSBC BANK',
        r'HUDSON NEWS?':'HUDSON NEWS',
        r'HUMAN RIGHTS CAMPAIGN|\bHRC\b':'HUMAN RIGHTS CAMPAIGN',
        r'IBEW':'IBEW UNION',
        r'ICELAND(IC)?\s?AIR':'ICELAND AIR',
        r'STATE DEM\w+ PARTY':'STATE DEMOCRATIC PARTY',
        r'STATE REP\w+ PARTY':'STATE REPUBLICAN PARTY',
        r'ILLINOIS LIFE (?:HEALTH)? INSURANCE':'ILLINOIS LIFE HEALTH INSURANC COUNCIL',
        r'^IMPACT\b':'IMPACT CONSULTING',
        r'IN N OUT':'IN N OUT',
        r'INTEGRATED SOLUTION':'INTERGRATED SOLUTIONS',
        r'INTELSAT': 'IN FLIGHT WIFI',
        r'INT?UIT': 'INTUIT',
        r'IPOSTAL':'IPOSTAL',
        r'ISTOCK':'ISTOCK PHOTO',
        r'J\s?P\s?MORGAN|JPM CHASE':'JP MORGAN',
        r'J\s?STREET':'JSTREET PAC',
        r'JET (?:ACCESS)?AVIATION':'JET AVIATION',
        r'JET\s?BLUE': 'JETBLUE AIRLINES',
        r'^JFK\b':'JFK AIRPORT',
        r'JOBS EDUCATION':'JOBS EDUCATION AND FAMILIES FIRST PAC',
        r'JOE\s?S PIZZA':'JOES PIZZA',
        r'JOE\s?S SEA':'JOES SEAFOOD',
        r'JOE\s?S STONE':'JOES STONE CRAB',
        r'JOE THE JUICE':'JOE THE JUICE',
        r'K L GATES': 'KL GATES',
        r'K R\b': 'KR BRANDING',
        r'KEY\s?BANK': 'KEY BANK',
        r'KIRWAN\s?S': 'KIRWANS',
        r'KLAVIYO': 'KLAVIYO',
        r'KRASON (?:AND )?WOOD': 'KRASON AND WOOD STRATEGY',
        r'L2': 'L2 INC',
        r'LA COLOMBE': 'LA COLOMBE',
        r'LA GRANDE? BOU': 'LA GRANDE BOUCHERIE',
        r'LA\s?QUINTA':'LA QUINTA HOTEL',
        r'LA\s?ARDENTE':'LA ARDENTE',
        r'LAGUARDIA|LGA\b':'LAGUARDIA AIRPORT',
        r'LARRY W POTTS':'LARRY W POTTS FOR NC HOUSE OF REPRESENTATIVES',
        r'LAX\b': 'LAX AIRPORT',
        r'LAZ\b': 'LAZ PARKING',
        r'LEGACY LIST':'LEGACY LISTS INC',
        r'LEGAL SEA\s?FOODS':'LEGAL SEAFOODS',
        r'LEX[I|U]S\s?NEX[I|U]S': 'LEXIS NEXIS',
        r'LGM\b':'LGMM CONSULTING',
        r'LIBERTARIAN MUTUAL AID':'LIBERTARIAN MUTUAL AID',
        r'LIBERTY MUTU?AL':'LIBERTY MUTUAL INSURANCE',
        r'LIGHTHOUSE (?:MEDIA )?PRODUCTIONS':'LIGHTHOUSE PRODUCTIONS',
        r'LINKEDIN':'LINKEDIN',
        r'LITTLE CEASARS':'LITTLE CEASARS PIZZA',
        r'LL\s?BEAN':'LL BEAN',
        r'LOEWS|LOWE S HOTEL':'LOEWS',
        r'LOGAN AIR':'LOGAN AIRPORT',
        r'LOGAN CIRCLE':'LOGAN CIRCLE GROUP',
        r'LOGICTECH':'LOGICTECH',
        r'LOTTE':'LOTTE HOTEL',
        r'LYFT':'LYFT',
        r'M (?:AND )?T\s?BANK': 'M AND T BANK',
        r'MAIL\s?CH[I|A]MP': 'MAILCHIMP',
        r'MAILER\sLITE':'MAILER LITE',
        r'MAJORITY COMMITTEE':'MAJORITY COMMITTEE',
        r'MAR A LAGO': 'MAR A LAGO',
        r'MARATHON (?:GAS|FUEL|PETROL)':'MARATHON GAS',
        r'MARGARITAVILLE':'MARGARITAVILLE',
        r'MASTERCARD':'MASTERCARD',
        r'MASTRO\s?S\b':'MASTROS',
        r'MBA (?:CONSULTING|DATA)':'MBA CONSULTING',
        r'MDI IMAG':'MDI IMAGING',
        r'MEETING STREET':'MEETING STREET RESEARCH',
        r'MEMBERS? DINING':'MEMBERS DINING',
        r'MERCANTILE':'MERCANTILE BANK',
        r'MERCHANT (?:BANK|CARD|SERVICE)': 'MERCHANT BANK',
        r'MGM\s?GRAND':'MGM GRAND HOTEL',
        r'MICROSOFT':'MICROSOFT',
        r'MIDDLE\s?SEAT':'MIDDLE SEAT CONSULTING',
        r'MINUTE\s?MAN':'MINUTE MAN PRESS',
        r'MISSIONs?WIRED': 'MISSIONWIRED',
        r'MOE S':'MOES SW GRILL',
        r'MOORE RESPONSE':'MOORE RESPONSE MANAGEMENT GROUP',
        r'MORTGAGE BANKERS':'MORTGAGE BANKERS PAC',
        r'MORTON\s?S':'MORTONS STEAKHOUSE',
        r'MOTHERSHIP (?:STRATEGIES)?':'MOTHERSHIP',
        r'MOVEON ORG':'MOVEON ORG',
        r'MSP AIR':'MSP AIRPORT',
        r'MTA':'NEW YORK SUBWAY',
        r'NAACP':'NAACP',
        r'NASW \w+ CHAPTER PACE': 'NATIONAL ASSOCIATION OF SOCIAL WORKERS',
        r'NATIONAL AIR TRAFF': 'NATIONAL AIR TRAFFIC CONTROLLERS ASSOCIATION',
        r'NATIONAL (?:RENTAL )?CAR\b':'NATIONAL CAR RENTAL',
        r'NATIONAL JOURNAL':'NATIONAL JOURNAL GROUP',
        r'NATIONAL REP\w+ CONG|\bNRCC\b':'NRCC',
        r'NATIONAL REP\w+ SEN|\bNRSC\b':'NRSC',
        r'NATIONAL RESEARCH':'NATIONAL RESEARCH INC',
        r'NATIONAL RESTA':'NATIONAL RESTAURANT ASSOCIATION',
        r'NATIONAL RIFLE|\bNRA\b':'NRA',
        r'NBC\s?UNIVERSAL':'NBC UNIVERSAL',
        r'NEW YORK TIME|NY\s?TIMES?': 'NEW YORK TIMES',
        r'NEWRK (?:LIBERTY )AIR': 'NEWARK AIRPORT',
        r'NEXT LEVEL': 'NEXT LEVEL PARTNERS',
        r'NGP\b|NGP\s?VAN|EVERYACTION':'NGP EVERYACTION',
        r'NRCC\b|NATIONAL REPUBLICAN CONGR': 'NRCC',
        r'NRDC\b|NATIONAL RESOURCE DEFENSE COM': 'NRDC',
        r'NRSC\n|NATION REPUBLICAN SEN\w+ COM': 'NRSC',
        r'NYC (?:\s?w\+\s?w\+?\s+)?(?:TAXI|CAB)': 'NYC TAXI',
        r'OFFICE (?:MAX )?DEPOT': 'OFFICE DEPOT',
        r'OMNI\b':'OMNI HOTELS',
        r'PACTION':'PACTION DATA',
        r'PANERA':'PANERA BREAD',
        r'PAPA JOHN':'PAPA JOHNS',
        r'PARAGON':'PARAGON SOLUTIONS',
        r'PATHFINDER':'PATHFINDER COMMUNICATIONS',
        r'PAY\s?CHEX':'PAYCHEX',
        r'PAYMENTECH': 'PAYMENTECH',
        r'PAY\s?PAL':'PAYPAL',
        r'PEET\sS':'PEETS COFFEE',
        r'PENDRY\b':'PENDRY HOTEL',
        r'PHONE2ACTION':'PHONE2ACTION',
        r'PIZZA|PIZZ[E|A]RIA':'PIZZA',
        r'PLANET\s?DIRECT':'PLANET DIRECT MAILING',
        r'PLANNED PARENTHOOD':'PLANNED PARENTHOOD',
        r'PNC\b': 'PNC BANK',
        r'POLITICAL COMPLIANCE':'POLITICAL COMPLIANCE MANAGEMENT',
        r'POLITICO\b':'POLITICO',
        r'POPEYE\s?S?':'POPEYES',
        r'POTBELLY':'POTBELLY',
        r'PRICELINE|PRICELN': 'PRICELINE',
        r'PRICEWATERHOUSE|PWC\b':'PWC',
        r'PRODUCTION (?:MANAGEMENT )?ONE': 'PRODUCTION ONE',
        r'PROPAY':'PROPAY',
        r'PUNCH\s?BOWL':'PUNCHBOWL',
        r'QDOBA':'QDOBA',
        r'QUICKBOOKS?':'QUICKBOOKS',
        r'QUORUM ANALYTICS':'QUORUM ANALYTICS',
        r'RAISING CANE':'RAISING CANES',
        r'REPUBLIC BANK':'REPUBLIC BANK',
        r'REVV?\b':'REVV FUNDRAISING',
        r'RIVER\s?SIDE\s?FM':'RIVERSIDE FM',
        r'ROTI\b':'ROTI',
        r'SONESTA':'ROYAL SONESTA',
        r'SAFEGAURD':'SAFEGAURD SHREDDING',
        r'SAFE\s?WAY':'SAFEWAY',
        r'SALES?\s?FORCE': 'SALESFORCE',
        r'SANDLER REIFF':'SANDLER REIFF LAMB ROSENSTEIN BIRKENSTOCK',
        r'SAPPHIRE':'SAPPHIRE STRATEGIES',
        r'SBUX|STAR\sBUCKS?':'STARBUCKS',
        r'SCR\b':'SCR ASSOCIATES',
        r'SEIU\b|SERVICE EMPLOYEES':'SEIU',
        r'SEQUOIA':'SEQUOIA RESEARCH',
        r'SFO\b':"SAN FRANSISCO AIRPORT",
        r'SHELL\b':'SHELL OIL',
        r'SHOPIFY':'SHOPIFY',
        r'SHRED\b':'SHREDDING SERVICE',
        r'SHUTTER\s?STOCK':'SHUTTERSTOCK',
        r'SIERRA CLUB':'SIERRA CLUB',
        r'SINCLAIR BROAD':'SINCLAIR BROADCASTING',
        r'SITX\b':'SIXT CAR RENTAL',
        r'SKDK':'SKD KNICKERBOCKER',
        r'SLACK T': 'SLACK',
        r'SNOOZE':'SNOOZE AM EATERY',
        r'SODEXO':'SODEXO',
        r'SOUTHWEST AIR': 'SOUTHWEST',
        r'SPEECHIFAI':'SPEECHIFAI',
        r'SPIRIT AIR':'SPIRIT',
        r'SPOT\s?HERO':'SPOTHERO',
        r'SPOTIFY': 'SPOTIFY',
        r'SQUARE\s?(?:CAPITAL|FEES|GROUP|INC)?':'SQUARE',
        r'SQUARE\s?SPACE':'SQUARESPACE',
        r'STAPLES':'STAPLES',
        r'STAR\s?LINK':'STARLINK',
        r'STATE FARM':'STATE FARM',
        r'STRATEGIC ADVANCE':'STRATEGIC ADVANCE',
        r'STRATEGIC PARTNERS':'STRATEGIC PARTNERS',
        r'STRIPE\b':'STRIPE',
        r'SUBSTACK\b':'SUBSTACK',
        r'SUNOCO\b':'SUNOCO',
        r'SUSAN B ANTHONY':'SUSAN B ANTHONY LIST',
        r'SWEET\s?GREEN':'SWEETGREEN',
        r'SWITCHBOARD':'SWITCHBOARD',
        r'SYNOV':'SYNOVUS BANK',
        r'T\s?MOBILE': 'TMOBILE',
        r'TANGO\s?CARD':'TANGO CARD',
        r'TACOS\s|TAQUERIA':'TACOS',
        r'TARGET\s?SMART':'TARGETSMART',
        r'TARGE\s?POINT':'TARGETPOINT',
        r'TARGET\b': 'TARGET STORES',
        r'TATANGO':'TATANGO',
        r'TATTE\b': 'TATTE',
        r'TAXI\b':'TAXI',
        r'TD\s?(?:BANK|CARD|MERCHANT)':'TD BANK',
        r'TEAMSTERS?':'TEAMSTERS UNION',
        r'UNITED PARCEL|UPS':'UPS',
        r'THRIFTY CAR':'THRIFTY CAR RENTAL',
        r'TIDES ADVOCACY':'TIDES ADVOCACY',
        r'TOTAL\s?WINE':'TOTAL WINE',
        r'TPUSA|TURNING POINT':'TURNING POINT USA',
        r'TRAVELERS':'TRAVELERS INSURANCE',
        r'TRIUMPH':'TRIUMP COMMUNICATIONS',
        r'TRUIST|SUN\s?TRUST':'TRUIST BANK',
        r'TRUMP (?:HOTEL|INTERNATIONAL|NATIONAL|RESTAURANTS|TOWER)': 'TRUMP HOTELS',
        r'TRUMP SAVE':'TRUMP SAVE AMERICA',
        r'TV\s?EYES': 'TV EYES',
        r'TWILIO':'TWILIO',
        r'TWITTER':'TWITTER',
        r'U\sHAUL':'UHAUL',
        r'U\s?S\s?BANK':'US BANK',
        r'UAW?|UNITED AUTO':'UAW',
        r'UBER\b':'UBER',
        r'UFCW':'UFCW',
        r'ULINE':'ULINE',
        r'UNION LEAGUE':'UNION LEAGUE',
        r'UNITE HERE':'UNITE HERE',
        r'UNITED AIR':'UNITED AIRLINES',
        r'UNITED (?:BANK|BUSINESS)': 'UNITED BANK',
        r'UNITED\s?HEALTH':'UNITED HEALTH CARE',
        r'UNITED POSTAL|USPS|POST OFFICE|POSTAL SER|US POST':'USPS',
        r'UPLAND':'UPLAND SOFTWARE',
        r'UPSWING':'UPSWING RESEARCH',
        r'UPWAVE':'UPWAVE DIGITAL',
        r'UPWORK':'UPWORK',
        r'US\s?BANK':'US BANK',
        r'MARSHALL?S? SERVICE':'US MARSHALS SERVICE',
        r'US SENATE':'US SENATE',
        r'US HOUSE': 'US HOUSE',
        r'USA TODAY':'USA TODAY',
        r'USAA':'USAA',
        r'VAIL (?:SKI|RESORT)':'VAIL RESORT',
        r'VANGAURD':'VANGAURD',
        r'VANTIV|WORLD\s?PAY':'VANTIV WORLDPAY',
        r'VENTRA':'VENTRA',
        r'VERIZON':'VERIZON',
        r'VFW':'VFW',
        r'VIASAT':'VIASAT IN FLIGHT',
        r'VIRGIN (?:ATLANTIC|HOTEL)':'VIRGIN ATLANTIC',
        r'VISA\b':'VISA',
        r'VISTA\s?PRINT':'VISTAPRINT',
        r'VOTE\s?VETS?':'VOTEVETS',
        r'VRBO':'VRBO',
        r'WAL\s?MART':'WALMART',
        r'WALL?\s?GREEN':'WALGREENS',
        r'WALL ST\w+ JOURN|WSJ':'WALL STREET JOURNAL',
        r'WASHINGTON POST':'WASHINGTON POST',
        r'WASTE (?:MANAGEMENT|MGMT)':'WASTE MANAGEMENT',
        r'WAVE\s?LENGTH':'WAVELEANGTH STRATEGIES',
        r'WA\s?WA':'WA WA',
        r'WAY TO LEAD':'WAY TO LEAD',
        r'WELLS?\s?FARGO':'WALLES FARGO',
        r'WELLS?\s?PRINT':'WELLS PRINT AND DIGITAL SERVICES',
        r'WENDY\s?S':'WENDYS',
        r'WESTERN NATIVE':'WESTERN NATIVE VOICE',
        r'WHITE HOUSE':'THE WHITE HOUSE',
        r'WHOLE\s?FOOD':'WHOLE FOODS',
        r'WINE\b':'WINE',
        r'WINSTON\s?S':'WINSTONS FLOWERS',
        r'WIX':'WIX',
        r'WOLF\s?GANG':'WOLFGANG PUCKS',
        r'WYHDHAM':'WYNDHAM HOTELS',
        r'YOTEL':'YOTEL',
        r'YOU\s?TUBE':'YOUTUBE',
        r'ZAPIER':'ZAPIER',
        r'ZENDESK':'ZENDESK',
        r'ZOOM':'ZOOM',
        r'REPUBLICAN PARTY OF \w{2}$':'STATE REPUBLICAN PARTY',
        r'LIBERTARIAN PARTY OF \w{2}$':'STATE LIBERTARIAN PARTY',
        r'DEMOCRATIC PARTY OF \w{2}$':'STATE DEMOCRATIC PARTY',
        r'\w{2} SECRETARY OF STATE':'STATE SECRETARY OF STATE',
        r'FRIENDS OF THE \d{2}TH WAR':'FRIENDS OF THE WARD',
        r'REPUBLICAN HOUSE CAUCUS|HOUSE REPUBLICAN CAUCUS':'STATE REPUBLICAN HOUSE CAUCUS',
        r'\w{2} BANKERS ASSOCIATION': 'STATE BANKERS ASSOCIATION',
        r'\w{2} REPUBLICAN SENATE CAUCUS|\w{2} SENATE REPUBLICAN CAUCUS':'STATE REPUBLICAN SENATE CAUCUS',
        r'DEM\w+ HOUSE CAUCUS|HOUSE DEM\w+ CAUCUS':'STATE DEMOCRATIC HOUSE CAUCUS',
        r'\w{2} DEM\w+ SENATE CAUCUS|\w{2} SENATE DEM\w+ CAUCUS':'STATE DEMOCRATIC SENATE CAUCUS',
        r'REPUBLICAN STATE COMMITTEE|STATE REPUBLICAN COMMITTEE':'STATE REPUBLICAN COMMITTEE',
        r'DEMOCRATIC STATE COMMITTEE|STATE DEMOCRATIC COMMITTEE':'STATE DEMOCRATIC COMMITTEE',
        r'HOUSE REPUBLICAN CAMPAIGN CMTE|REPUBLICAN HOUSE CAMPAIGN CMTE':'STATE REPUBLICAN CAMPAIGN CMTE'
    }

    for pattern, vendor in vendor_name_dict.items():
        if re.search(pattern, input_str, re.IGNORECASE):
            return vendor  # return single mapped vendor name
    return None


In [20]:
# Create a vendor column based on the mappings above
spending_data['vendor'] = spending_data['payee_id'].apply(lambda x: parse(str(x)) if pd.notnull(x) else None)

In [21]:
spending_data.head()

,form_type,filer_committee_id_number,payee_organization_name,payee_last_name,payee_first_name,expenditure_date,expenditure_amount,expenditure_purpose_descrip,beneficiary_candidate_last_name,beneficiary_candidate_first_name,memo_code,memo_text_description,form_type_desc,payee_name,beneficiary_cand_name,payee_id,vendor
0,SB23,C00450965,MIKE BOST FOR CONGRESS COMMITTEE,NaN,NaN,,1000.0,CONTRIBUTION TO COMMITTEE,BOST,MIKE,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,MIKE BOST,MIKE BOST FOR CONGRESS COMMITTEE,None
1,SB23,C00450965,ANNIE,NaN,NaN,,2500.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,ANNIE,None
2,SB23,C00450965,BRETTPAC,NaN,NaN,,1500.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,BRETTPAC,BRETTPAC
3,SB23,C00450965,CASEY KEYSTONE VICTORY FUND,NaN,NaN,,1000.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,CASEY KEYSTONE VICTORY FUND,None
4,SB23,C00450965,OORAH,NaN,NaN,,2500.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,OORAH,None


In [ ]:
# See remaining payee ids
payees = pd.DataFrame(spending_data[spending_data['vendor'].isna()].payee_id.unique())

payees.rename(columns={payees.columns[0]: "payee_id"}, inplace= True)

payees.sort_values(by ='payee_id')

In [23]:
# Calculate the dimensionality reduction so far
n_vendors = len(spending_data.vendor.unique()) + len(spending_data[spending_data['vendor'].isna()].payee_id.unique())

1-(n_vendors/n_original_payees)


0.08587745287474768

## Cleaning Contribution Expenditures 
To clean contribution expenditures I am going to use the committee master dataset from the FEC's Bulk Data site to try and fuzzy match committees to their official names.
To do this i need to do the same state name cleaning I did for the payee id so state names become state abbreviations


In [24]:
cmte = pd.read_csv('../data/raw/cmte_master_24.csv')
cmte.head()

,CMTE_ID,CMTE_NM,TRES_NM,CMTE_ST1,CMTE_ST2,CMTE_CITY,CMTE_ST,CMTE_ZIP,CMTE_DSGN,CMTE_TP,CMTE_PTY_AFFILIATION,CMTE_FILING_FREQ,ORG_TP,CONNECTED_ORG_NM,CAND_ID
0,C00000059,"HALLMARK CARDS, INC. PAC (HALLPAC)","KLEIN, CASSIE MS.","2501 MCGEE, MD853",NaN,KANSAS CITY,MO,64108,B,Q,UNK,M,C,"HALLMARK CARDS, INC.",NaN
1,C00000422,AMERICAN MEDICAL ASSOCIATION POLITICAL ACTION ...,"JORDAN, JOHN ROBERT MR.","25 MASSACHUSETTS AVE, NW",SUITE 600,WASHINGTON,DC,200017400,B,Q,NaN,M,NaN,AMERICAN MEDICAL ASSOCIATION,NaN
2,C00000489,D R I V E POLITICAL FUND CHAPTER 886,JERRY SIMS JR,3528 W RENO,NaN,OKLAHOMA CITY,OK,73107,U,N,NaN,Q,L,NaN,NaN
3,C00000729,AMERICAN DENTAL ASSOCIATION POLITICAL ACTION C...,"TIPPETT-WHYTE, JUDEE DR.","400 C STREET, NE",NaN,WASHINGTON,DC,20002,B,Q,UNK,M,M,INDIANA DENTAL PAC,NaN
4,C00000885,INTERNATIONAL UNION OF PAINTERS AND ALLIED TRA...,"SMITH, GREGG",7234 PARKWAY DRIVE,NaN,HANOVER,MD,21076,B,Q,UNK,M,L,INTERNATIONAL UNION OF PAINTERS AND ALLIED TRADES,NaN


In [25]:
# Apply same string cleaning processes 
cmte = normalize_string_columns(cmte) 
cmte['CMTE_NM'] =  cmte['CMTE_NM'].fillna('').apply(replace_state_abbrev)
cmte['cmte_nm_len'] = cmte['CMTE_NM'].apply(lambda x: len(x))



In [26]:
cmte_vendors = spending_data[(spending_data['form_type'].isin(['SB22','SB23','SB29'])) & (spending_data['vendor'].isna()) & ((spending_data['payee_id'].str.len()>10) | (spending_data['payee_id'].str.contains(r'\w+PAC$', regex= True, case = False)))].payee_id.unique().tolist()

In [27]:
cmte_vendors = sorted(cmte_vendors, key=len)
cmte_vendors

['MPAC',
 'CPAC',
 'MCPAC',
 'PBPAC',
 'WEPAC',
 'HAPAC',
 'GOPAC',
 'BOPAC',
 'FFPAC',
 'BIPAC',
 'FEPAC',
 'ADPAC',
 'LIPAC',
 'IIPAC',
 'PIPAC',
 'SIPAC',
 'CEPAC',
 'RVFPAC',
 'BEEPAC',
 'ARKPAC',
 'BOWPAC',
 'KEYPAC',
 'ZENPAC',
 'WAXPAC',
 'BATPAC',
 'AEMPAC',
 'LEGPAC',
 'KEVPAC',
 'BECPAC',
 'PABPAC',
 'GASPAC',
 'MORPAC',
 '150PAC',
 'CARPAC',
 'CAMPAC',
 'PAWPAC',
 'FEDPAC',
 'RFWPAC',
 'OCCPAC',
 'MADPAC',
 'NVRPAC',
 'CBCPAC',
 'WINPAC',
 'ICBPAC',
 'JEBPAC',
 'BIZPAC',
 'NORPAC',
 'BACKPAC',
 'LANKPAC',
 'ALSOPAC',
 'THOMPAC',
 'JACKPAC',
 'AR RPAC',
 'DE RPAC',
 'FL RPAC',
 'GA RPAC',
 'KY RPAC',
 'MD RPAC',
 'MI RPAC',
 'MT RPAC',
 'NC RPAC',
 'NH RPAC',
 'NJ RPAC',
 'SD RPAC',
 'TN RPAC',
 'TX RPAC',
 'WA RPAC',
 'WV RPAC',
 'PHILPAC',
 'CITIPAC',
 'FOODPAC',
 'REITPAC',
 'MO RPAC',
 'UT RPAC',
 'FAIRPAC',
 'WPBTPAC',
 'CMCCPAC',
 'VIEWPAC',
 'DOUGPAC',
 'NC MPAC',
 'LA RPAC',
 'OR RPAC',
 'VINEPAC',
 'IL MPAC',
 'WOLFPAC',
 'MN RPAC',
 'AL RPAC',
 'COALPAC',
 'BANKPAC'

To clean the contributions to PACs and committees I attempt to use fuzzy matching to create groups that we can assign the same name to. To start I find the best match across all metric types and save them as a data frame. This helped me to see how matching could go. Importantly, I'm not comparing the payee_ids against each other, but the official committee names from the FEC committee master dataset.


In [28]:
# Fuzzy matching exploration 
cmte_names = cmte[((cmte['cmte_nm_len']>10) | (cmte['CMTE_NM'].str.contains(r'\w+PAC$'))) & (~cmte['CMTE_NM'].isin(['VICTORY FUND','ACTION FUND','POLITICAL ACTION COMMITTEE']))].CMTE_NM.unique().tolist()
to_cat = cmte_vendors


def get_best_matches(list_1,list_2):
    results = []
    for aa in list_1:
        results_dict = {"vendor": aa}

        for score_type, scorer in [
            ('ratio',fuzz.ratio),
            ('partial_ratio', fuzz.partial_ratio),
            ('token_sort_ratio', fuzz.token_sort_ratio),
            ('token_set_ratio', fuzz.token_set_ratio)
        ]:
            best_match = process.extractOne(aa, list_2, scorer = scorer)
            if best_match:
                results_dict[f"{score_type}_match"] = best_match[0]
                results_dict[f"{score_type}_score"] = best_match[1]
            else:
                results_dict[f"{score_type}_match"] = None
                results_dict[f"{score_type}_score"] = None
        
        levenshtein_best = min(list_2, key = lambda x: distance.Levenshtein.distance(aa, x))
        levenshtein_score = distance.Levenshtein.distance(aa, levenshtein_best)
        results_dict['levenshtein_match'] = levenshtein_best
        results_dict['levenshtein_score'] = levenshtein_score

        results.append(results_dict)
    
    return results

matches = get_best_matches(to_cat,cmte_names)
matches_df = pd.DataFrame(matches)


In [29]:
matches_df.head(25)

,vendor,ratio_match,ratio_score,partial_ratio_match,partial_ratio_score,token_sort_ratio_match,token_sort_ratio_score,token_set_ratio_match,token_set_ratio_score,levenshtein_match,levenshtein_score
0,MPAC,IMPAC,88.888889,UNITED STATES TELECOM ASSOCIATION POLITICAL AC...,100.000000,IMPAC,88.888889,MARATHON PETROLEUM CORPORATION EMPLOYEES POLIT...,100.000000,KPAC,1
1,CPAC,CPAC,100.000000,ACPAC ACA INTERNATIONAL POLITICAL ACTION COMMI...,100.000000,CPAC,100.000000,CPAC ACTION,100.000000,CPAC,0
2,MCPAC,MSCPAC,90.909091,REDWINGMCPAC,100.000000,MSCPAC,90.909091,MSCPAC,90.909091,MSCPAC,1
3,PBPAC,BOPAC,80.000000,NATIONAL ASSOCIATION OF BROADCASTERS POLITICAL...,88.888889,BOPAC,80.000000,BOPAC,80.000000,LBPAC,1
4,WEPAC,EDPAC,80.000000,WOMEN FOR EQUALITY PAC WEPAC,100.000000,EDPAC,80.000000,WOMEN FOR EQUALITY PAC WEPAC,100.000000,ARPAC,2
5,HAPAC,HPAC,88.888889,THE HOSPITAL AND HEALTHSYSTEM ASSOCIATION OF P...,100.000000,HPAC,88.888889,THE HOSPITAL AND HEALTHSYSTEM ASSOCIATION OF P...,100.000000,HPAC,1
6,GOPAC,GIPAC,80.000000,GOPAC ELECTION FUND,100.000000,GIPAC,80.000000,GOPAC ELECTION FUND,100.000000,GIPAC,1
7,BOPAC,BOPAC,100.000000,BOPAC,100.000000,BOPAC,100.000000,BOPAC,100.000000,BOPAC,0
8,FFPAC,RVFPAC,72.727273,CHURCHILL DOWNS INCORPORATED FPAC,88.888889,RVFPAC,72.727273,RVFPAC,72.727273,ARPAC,2
9,BIPAC,GIPAC,80.000000,BIPARTISAN POLITICAL ACTION COMMITTEE THE BANK...,100.000000,GIPAC,80.000000,BIPARTISAN POLITICAL ACTION COMMITTEE THE BANK...,100.000000,GIPAC,1


In [30]:
# the state party and district wards are often only off by a letter or two, but the differences are significant and I don't want to misattribute an expenditure. So i want to look more closely at these groups
state_party_patterns = r'\b(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY) (?:DEM\w+|REPU\w+|HOUSE DEM\w+|SENATE DEMw\+|HOUSE REPU\w+|SENATE REPUw\+|COUNTY|DFL)'
district_ward_patterns = r'\d{2}TH|1ST'

state_party_vendors = sorted(matches_df[(matches_df['vendor'].str.contains(state_party_patterns, regex=True))].vendor.unique().tolist())
district_ward_vendors = sorted(matches_df[(matches_df['vendor'].str.contains(district_ward_patterns, regex=True))].vendor.unique().tolist())


/var/folders/cl/bdrh3m8j0fdfj6zm7zhhc2ch0000gn/T/ipykernel_76724/515219276.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  state_party_vendors = sorted(matches_df[(matches_df['vendor'].str.contains(state_party_patterns, regex=True))].vendor.unique().tolist())


In [31]:
# Create a boolean column when all match values are the same
matches_df['same_match'] = (
    (matches_df['ratio_match'] == matches_df['partial_ratio_match']) &
    (matches_df['partial_ratio_match'] == matches_df['token_sort_ratio_match']) &
    (matches_df['token_sort_ratio_match'] == matches_df['token_set_ratio_match']) &
    (matches_df['token_set_ratio_match'] == matches_df['levenshtein_match'])
)

cols = [
    'ratio_score',
    'partial_ratio_score',
    'token_sort_ratio_score',
    'token_set_ratio_score'
]

# Average the match scores across all metrics
matches_df['avg_score'] = matches_df[cols].mean(axis = 1)


In [ ]:
# Attempting to find a rule that could help assign matches
matches_df[
    (matches_df['same_match']==True) 
    & (matches_df['levenshtein_score']>0) 
    & ~(matches_df['vendor'].str.contains(state_party_patterns, regex = True)) 
    & ~(matches_df['vendor'].str.contains(district_ward_patterns, regex = True))
    & ~(matches_df['ratio_match'].str.contains(state_party_patterns, regex = True)) 
    & ~(matches_df['ratio_match'].str.contains(district_ward_patterns, regex = True))
    ].sort_values('avg_score',ascending = False)



### Methodology 2 

Select a single metric from rapidfuzz to try and assign matches. I chose token_sort_ratio because I don't want the order of tokens in a string to matter, but the more tokens that match the better. 

I use a high match threshold (95) because I am very averse to misattributing spending. 

In [33]:
unique_names = cmte_vendors
threshold = 95
groups = []

while unique_names:
    name = unique_names.pop(0)
    # Find all similar names
    matches = process.extract(name, unique_names, scorer=fuzz.token_sort_ratio, score_cutoff=threshold)
    # Add name itself
    group = [name] + [m[0] for m in matches]
    groups.append(group)
    # Remove matched names
    unique_names = [x for x in unique_names if x not in [m[0] for m in matches]]

groups


[['MPAC'],
 ['CPAC'],
 ['MCPAC'],
 ['PBPAC'],
 ['WEPAC'],
 ['HAPAC'],
 ['GOPAC'],
 ['BOPAC'],
 ['FFPAC'],
 ['BIPAC'],
 ['FEPAC'],
 ['ADPAC'],
 ['LIPAC'],
 ['IIPAC'],
 ['PIPAC'],
 ['SIPAC'],
 ['CEPAC'],
 ['RVFPAC'],
 ['BEEPAC'],
 ['ARKPAC'],
 ['BOWPAC'],
 ['KEYPAC'],
 ['ZENPAC'],
 ['WAXPAC'],
 ['BATPAC'],
 ['AEMPAC'],
 ['LEGPAC'],
 ['KEVPAC'],
 ['BECPAC'],
 ['PABPAC'],
 ['GASPAC'],
 ['MORPAC'],
 ['150PAC'],
 ['CARPAC'],
 ['CAMPAC'],
 ['PAWPAC'],
 ['FEDPAC'],
 ['RFWPAC'],
 ['OCCPAC'],
 ['MADPAC'],
 ['NVRPAC'],
 ['CBCPAC'],
 ['WINPAC'],
 ['ICBPAC'],
 ['JEBPAC'],
 ['BIZPAC'],
 ['NORPAC'],
 ['BACKPAC'],
 ['LANKPAC'],
 ['ALSOPAC'],
 ['THOMPAC'],
 ['JACKPAC'],
 ['AR RPAC'],
 ['DE RPAC'],
 ['FL RPAC'],
 ['GA RPAC'],
 ['KY RPAC'],
 ['MD RPAC'],
 ['MI RPAC'],
 ['MT RPAC'],
 ['NC RPAC'],
 ['NH RPAC'],
 ['NJ RPAC'],
 ['SD RPAC'],
 ['TN RPAC'],
 ['TX RPAC'],
 ['WA RPAC'],
 ['WV RPAC'],
 ['PHILPAC'],
 ['CITIPAC'],
 ['FOODPAC'],
 ['REITPAC'],
 ['MO RPAC'],
 ['UT RPAC'],
 ['FAIRPAC'],
 ['WPBTPAC'],
 [

In [34]:
# Isolate groups with more than one value
groups_n_2 = [group for group in groups if len(group)>1]
len(groups_n_2)


1619

In [35]:
# Isolate names that did not have any matches given the threshold 
groups_n_1 = [group for group in groups if len(group)==1]
len(groups_n_1)

32750

In [36]:
# For groups >=2, assign a group name that is equal to the shortest string in the group. This group name will be used to map payees to a vendor 
mapping = {}

# Assign a group name to the groups to use in vendor definition 
for ii in groups_n_2:
    df = pd.DataFrame({'item': ii})
    df['st_len'] = df['item'].apply(lambda s: len(s))
    df = df.sort_values('st_len')
    name = df.iloc[0,0]
    mapping[name] = ii

mapping


{'VA BANKPAC': ['VA BANKPAC', 'VBA BANKPAC'],
 'A BOLDER FL': ['A BOLDER FL', 'A BOULDER FL'],
 'JOHN CURTIS': ['JOHN CURTIS', 'CURTIS JOHN'],
 'DEBONA JOHN': ['DEBONA JOHN', 'JOHN DEBONA'],
 'HALL STEVEN': ['HALL STEVEN', 'STEVEN HALL'],
 'JACKY ROSEN': ['JACKY ROSEN', 'ROSEN JACKY'],
 'ANGIE CRAIG': ['ANGIE CRAIG', 'CRAIG ANGIE'],
 'MOORE BLAKE': ['MOORE BLAKE', 'MOORRE BLAKE'],
 'BERLINROSEN': ['BERLINROSEN', 'BERLIN ROSEN'],
 'KIM SCHRIER': ['KIM SCHRIER', 'SCHRIER KIM'],
 'DADE PHELAN': ['DADE PHELAN', 'PHELAN DADE'],
 'VOTE BOLICK': ['VOTE BOLICK', 'VOTE BOLLICK'],
 'PRESTA LISA': ['PRESTA LISA', 'LISA PRESTA'],
 'TREASURE FL': ['TREASURE FL', 'TREASURER FL'],
 'MARK LEVINE': ['MARK LEVINE', 'LEVINE MARK'],
 'LOPEZ VICKI': ['LOPEZ VICKI', 'LOPEZ VICKIE'],
 'LER BRANDON': ['LER BRANDON', 'LEHR BRANDON'],
 'PA TOGETHER': ['PA TOGETHER', 'TOGETHER PA'],
 'OWEN FOR UT': ['OWEN FOR UT', 'OWENS FOR UT'],
 'FOX WHITNEY': ['FOX WHITNEY', 'WHITNEY FOX'],
 'JIMMY JOHNS': ['JIMMY JOHNS', 'J

In [37]:

# reconfigure mapping to be pattern: group instead of group: pattern 

def convert_to_regex_map(original_dict):
    regex_map = {}

    for name, variants in original_dict.items():
        # Escape each variant for regex safety
        escaped_variants = [v for v in variants]

        # Join into one regex pattern
        regex_pattern = r"|".join(escaped_variants)

        # Map pattern → canonical key
        regex_map[regex_pattern] = name

    return regex_map


mapping_2 = convert_to_regex_map(mapping)

In [38]:
# Compile regex map so that we can use it to assign payees to vendors 
def compile_regex_map(regex_map):
    compiled = []
    for pattern, replacement in regex_map.items():
        compiled.append((re.compile(pattern, re.IGNORECASE), replacement))
    return compiled


mapping_2_sorted = dict(
    sorted(mapping_2.items(), key=lambda x: -len(x[0]))
)


In [39]:
# apply group names as vendors to matching payee ids 
compiled_map = compile_regex_map(mapping_2)

def apply_regex_mapping(text, regex_map):
    if pd.isna(text):
        return None
    for regex, replacement in compiled_map:
        if regex.search(text):
            return replacement
    return None

# identify indexes of na vendor rows 
mask = spending_data['vendor'].isna()

# Apply mapping to null vendors 
spending_data.loc[mask, 'vendor'] = spending_data.loc[mask, 'payee_id'].apply(
    lambda x: apply_regex_mapping(x, mapping_2)
)


In [40]:
spending_data.head()

,form_type,filer_committee_id_number,payee_organization_name,payee_last_name,payee_first_name,expenditure_date,expenditure_amount,expenditure_purpose_descrip,beneficiary_candidate_last_name,beneficiary_candidate_first_name,memo_code,memo_text_description,form_type_desc,payee_name,beneficiary_cand_name,payee_id,vendor
0,SB23,C00450965,MIKE BOST FOR CONGRESS COMMITTEE,NaN,NaN,,1000.0,CONTRIBUTION TO COMMITTEE,BOST,MIKE,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,MIKE BOST,MIKE BOST FOR CONGRESS COMMITTEE,MIKE BOST FOR CONGRESS COMMITT
1,SB23,C00450965,ANNIE,NaN,NaN,,2500.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,ANNIE,None
2,SB23,C00450965,BRETTPAC,NaN,NaN,,1500.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,BRETTPAC,BRETTPAC
3,SB23,C00450965,CASEY KEYSTONE VICTORY FUND,NaN,NaN,,1000.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,CASEY KEYSTONE VICTORY FUND,None
4,SB23,C00450965,OORAH,NaN,NaN,,2500.0,CONTRIBUTION TO COMMITTEE,NaN,NaN,NaN,NaN,CONTRIBUTIONS TO CANDIDATES AND COMMITTEES,,,OORAH,None


In [41]:
# recalculate dimentionality reduction
n_vendors = len(spending_data.vendor.unique()) + len(spending_data[spending_data['vendor'].isna()].payee_id.unique())

1-(n_vendors/n_original_payees)

0.11991355003077275

In [42]:
# I looked up the largest uncoded payees by number of lineitems and coded them manually 
mapping_3 = {
    # just Ted Cruz because he has multiple committees, pres, senate, etct
    r'TED CRUZ': 'TED CRUZ',
    r'ANEDOT': 'ANEDOT',
    r'BERNIE MORENO': 'BERNIE MORENO FOR SENATE',
    r'RICK SCOTT FOR':'RICK SCOTT FOR FL',
    r'JOSH HAWLEY':'JOSH HAWLEY FOR SENATE',
    r'KARI LAKE':'KARI LAKE FOR SENATE',
    r'ELISSA SLOTKIN': 'ELISSA SLOTKIN FOR SENATE',
    r'ANGELA ALSOBROOKS':'ANGELA ALSOBROOKS FOR SENATE',
    r'TAMMY BALDWIN': 'TAMMY BALDWIN FOR SENATE',
    r'MARY PELTOLA': 'MARY PELTOLA FOR SENATE',
    r'ROSEN FOR NV| JACKIE ROSEN': 'ROSEN FOR NV'
}
compiled_map = compile_regex_map(mapping_3)

mask = spending_data['vendor'].isna()

spending_data.loc[mask, 'vendor'] = spending_data.loc[mask, 'payee_id'].apply(
    lambda x: apply_regex_mapping(x, mapping_3)
)

In [43]:
# Recalculate dimensionality reduction
n_vendors = len(spending_data.vendor.unique()) + len(spending_data[spending_data['vendor'].isna()].payee_id.unique())

1-(n_vendors/n_original_payees)

0.12008530493652225

In [44]:
# I need to move on. Creating vendor_full as the concatenation of vendor and payee id 
spending_data['vendor_full'] = (
    spending_data[['vendor', 'payee_id']]
    .replace('', pd.NA)         
    .bfill(axis=1)              
    .iloc[:, 0]                 
)


In [45]:
# final dimensionality reduction assessment
n_test = len(spending_data['payee_id'].unique())
n_vendors = len(spending_data['vendor_full'].unique())

print('I have reduced dimensionality by',1-(n_vendors/n_test))

I have reduced dimensionality by 0.10710342823939567


In [46]:
# Reading in the PACs dataset
pacs = pd.read_csv('../data/processed/pac_financials.csv', usecols= ['committee_id', 
'committee_name',
'committee_designation',
'committee_designation_full', 
'committee_type', 
'committee_type_full',
'filing_frequency', 
'organization_type',
'organization_type_full', 
'treasurer_name',
'committee_state', 
'coverage_end_date', 
'coverage_start_date', 
'first_file_date', 
'cycle',
'receipts',
'individual_contributions', 
'individual_itemized_contributions',
'individual_unitemized_contributions',
'disbursements'
])

pacs.head()

,committee_designation,committee_designation_full,committee_id,committee_name,committee_state,committee_type,committee_type_full,coverage_end_date,coverage_start_date,cycle,disbursements,filing_frequency,first_file_date,individual_contributions,individual_itemized_contributions,individual_unitemized_contributions,organization_type,organization_type_full,receipts,treasurer_name
0,U,Unauthorized,C00000489,D R I V E POLITICAL FUND CHAPTER 886,OK,N,PAC - Nonqualified,2024-12-31T00:00:00,2023-01-01T00:00:00,2024,27426.58,Q,1976-07-13,0.00,0.00,0.00,L,Labor Organization,93817.80,JERRY SIMS JR
1,B,Lobbyist/Registrant PAC,C00000729,AMERICAN DENTAL ASSOCIATION POLITICAL ACTION C...,DC,Q,PAC - Qualified,2024-12-31T00:00:00,2023-01-01T00:00:00,2024,2617456.75,M,1975-02-09,2118774.56,1304725.93,814048.63,M,Membership Organization,2228046.98,"TIPPETT-WHYTE, JUDEE DR."
2,B,Lobbyist/Registrant PAC,C00000885,INTERNATIONAL UNION OF PAINTERS AND ALLIED TRA...,MD,Q,PAC - Qualified,2024-12-31T00:00:00,2023-01-01T00:00:00,2024,3244535.22,M,1975-03-20,5306388.90,405469.89,4900919.01,L,Labor Organization,5525285.55,"SMITH, GREGG"
3,B,Lobbyist/Registrant PAC,C00000059,"HALLMARK CARDS, INC. PAC (HALLPAC)",MO,Q,PAC - Qualified,2024-12-31T00:00:00,2023-01-01T00:00:00,2024,117500.00,M,1976-04-22,91727.18,62548.68,29178.50,C,Corporation,95727.18,"KLEIN, CASSIE MS."
4,B,Lobbyist/Registrant PAC,C00000422,AMERICAN MEDICAL ASSOCIATION POLITICAL ACTION ...,DC,Q,PAC - Qualified,2024-12-31T00:00:00,2023-01-01T00:00:00,2024,2621577.12,M,1975-03-06,1666298.01,1300778.67,365519.34,NaN,NaN,2096337.89,"JORDAN, JOHN ROBERT MR."


In [47]:
pacs.columns

Index(['committee_designation', 'committee_designation_full', 'committee_id',
       'committee_name', 'committee_state', 'committee_type',
       'committee_type_full', 'coverage_end_date', 'coverage_start_date',
       'cycle', 'disbursements', 'filing_frequency', 'first_file_date',
       'individual_contributions', 'individual_itemized_contributions',
       'individual_unitemized_contributions', 'organization_type',
       'organization_type_full', 'receipts', 'treasurer_name'],
      dtype='object')

### Code Expenditure types

Here I try to repeat the work done above on the expenditure types. Ultimately, in the interest of time I decided to keep the expense coding simple: contributions or transfers to candidates or committees or not. 

In [48]:
# To start, I am coding expenditure data based on their form_types, either contributions or transfers, or not. 

spending_data['exp_cat'] = np.where(spending_data['form_type']=='SB23', 'Contribution to Candidate or Committee',
                           np.where(spending_data['form_type']=='SB22', 'Transfer to affiliated committee',
                           'Other'
      )
)


When experimenting with expenditure purpose cleaning, I use a lower threshold becuase the concerns about mis coding are less acute. Ultimately I chose not to focus resources on cleaning expenditure purpose descriptions for this project, but in the future, this work will be relevant when trying to classify the other expenditures.

In [49]:
unique_purposes = spending_data['expenditure_purpose_descrip'].unique().tolist()
threshold = 85
purpose_groups = []

while unique_purposes:
    purpose = unique_purposes.pop(0)
    # Find all similar purposes
    matches = process.extract(purpose, unique_purposes, scorer=fuzz.token_sort_ratio, score_cutoff=threshold)
    # Add purpose itself
    group = [purpose] + [m[0] for m in matches]
    purpose_groups.append(group)
    # Remove matched purposes
    unique_purposes = [x for x in unique_purposes if x not in [m[0] for m in matches]]

purpose_groups




[['CONTRIBUTION TO COMMITTEE',
  'CONTRIBUTION TO COMMMITTEE',
  'CONTRIBUTON TO COMMITTEE',
  'CONTRIBUTION T COMMITTEE',
  'CONTIBUTION TO COMMITTEE',
  'CONTRIBUTION TO COMMITEE'],
 [nan],
 ['IN KIND LEGAL'],
 ['PAC POLITICAL CONTRIBUTION',
  'PAC POLTICAL CONTRIBUTION',
  'PAC POLITCAL CONTRIBUTION',
  'PAC POLITICAL CONTRIBTION',
  'PAC POLITICAL CONTRIBUTIN',
  'PAC POLITICAL CONRIBUTION'],
 ['PAC COMPLIANCE', 'PAC COMPLAINCE', 'JFC COMPLIANCE', 'FEC COMPLIANCE'],
 ['PAC FUNDRAISING',
  'PAC FUNDRASING',
  'PAC SMS FUNDRAISING',
  'MARCH  FUNDRAISING',
  'JFC FUNDRAISING',
  'MAY  FUNDRAISING'],
 ['CONTRIBUTION',
  'CPONTRIBUTION',
  'CONTRIBUTIUON',
  'CONTRIBUTIONB',
  'CONTRIBUTIOIN',
  'CONTRIBUTRION'],
 ['PRIMARY', 'PIMARY'],
 ['MERCHANT FEES',
  'MERCHANT FEE',
  'MECHANT FEES',
  'MERCHANT FES',
  'E MERCHANT FEES',
  'MERCAHNT FEES'],
 ['BANK FEE', 'BANK FEES', 'BANK FREE', 'BANKFEE', 'BANK FE', 'BANKT FEES'],
 ['CONTRIBUTION TO A STATE COMMITTEE',
  'CONTRIBUTION TO STAT

In [ ]:
# Fill all remaining unclassified categories as other
spending_data['exp_cat']=spending_data['exp_cat'].fillna('other').str.upper()

## Aggregate the spending data for use in the Network visualization

In [ ]:
spending_data_agg = (
    spending_data
    .groupby(['filer_committee_id_number', 'vendor_full','exp_cat'])
    # calculate the total amount spent by a PAC on a vendor and the # of payments
    .agg(
        ttl_spend=('expenditure_amount', 'sum'),
        n_transactions=('vendor', 'size')
    )
    .reset_index()
)

# merge with pac qualitative data to get info like cmte_type and cmte_designation
spending_data_agg = spending_data_agg.merge(pacs, left_on ='filer_committee_id_number',right_on = 'committee_id', how = 'left')

spending_data_agg.head()


### Add Independent expenditure data to spending_data_add

In [ ]:
# load SE data
se = pd.read_csv('../data/raw/SE_dataset.csv')

#se.head()

# Apply same string cleaning as above
se = normalize_string_columns(se) 

#se.isna().sum()

#se[(se['candidate_id_number'].isna())].filter(['candidate_last_name','candidate_first_name']).drop_duplicates().sort_values('candidate_last_name')

# aggregate SE data 
se_data_agg = (
    se
    .groupby(['filer_committee_id_number'])
    .agg(
        ttl_spend=('expenditure_amount', 'sum'),
        n_transactions = ('filer_committee_id_number','size')
    )
    .reset_index()
)

# merge in pac information to have similar columns
se_data_agg = se_data_agg.merge(pacs, left_on ='filer_committee_id_number',right_on = 'committee_id', how = 'left')
se_data_agg['vendor_full'] = 'INDEPENDENT EXPENDITURE'
se_data_agg['exp_cat'] = 'Independent Expenditure'

se_data_agg.head()


In [ ]:
# make sure columns are the same for binding the data together
print(len(spending_data_agg.columns))
print(len(se_data_agg.columns))

In [ ]:
# Bind SE data with SB data 
spending_data_agg = pd.concat([spending_data_agg,se_data_agg], ignore_index= True)

spending_data_agg.exp_cat.unique()

### Define calculated columns to use in filters

In [ ]:
#Define calculated columns 

# pct_indiv: the percent of a PACs receipts that come from individuals
spending_data_agg['pct_indiv'] = spending_data_agg['individual_contributions']/spending_data_agg['receipts']

# pct_unitem: the percent of a PACs receipts that come from unitemized individuals
spending_data_agg['pct_unitem'] = spending_data_agg['individual_unitemized_contributions']/spending_data_agg['receipts']

# n_unique_filers: the number of filers that paid a given vendor
spending_data_agg['n_unique_filers'] = (
    spending_data_agg.groupby('vendor_full')['filer_committee_id_number']
    .transform('nunique')
)

# n_unique_vendors: the number of vendors that a given filer used 
spending_data_agg['n_unique_vendors'] = (
    spending_data_agg.groupby('filer_committee_id_number')['vendor_full']
    .transform('nunique')
)

union_pattern = r'FEDERATION|UNION|LOCAL (?:#|NO\.)?\s?\d+|ASSOC.?\w+? OF|SEIU|IBEW|UAW|TEAMSTERS|BROTHERHOOD OF|UNITED \w+ WORKERS OF|AFSCME|VOLUNTARY POLITICAL COMMITTEE'

spending_data_agg['is_union_pac'] = np.where( spending_data_agg.organization_type_full.isin(['Labor Organization','Trade Association','Membership Organization','Cooperative']), True, 
                              np.where( (spending_data_agg.organization_type_full.isna() & spending_data_agg.committee_name.str.contains(union_pattern,regex=True)),True, False))

spending_data_agg['pct_ttl_spend'] = np.where(
    spending_data_agg['ttl_spend'] > 0,
    100*(spending_data_agg['ttl_spend']/spending_data_agg['disbursements']),
    0.01        
)


### Export final dataset 

In [ ]:

spending_data_agg.to_csv("../data/processed/pac_vendor_agg.csv", index=False)

## Earlier Work

All of the work below represents attempts at other text mining / clustering methods. Ultimately, given my high aversion to mis attributing expenditures, I decided to not use more common text clustering methods in favor of the rapidfuzz and regex text cleaning.

To start, I decided to try HDBSCAN because I didn't want to set the number of clusters, and the max_cluster_size parameter was a helpful tool for this work. I would not expect many clusters should have more than 5 values, because we are mostly looking to do text cleaning. 

Ultimately, some of the clusters look great, some are complicated by candidates with similar names running in different states or candidate names and staff or consultant names being similar, others are way off. Also, an issue I ran into with more simple text mining for text cleaning, since naming conventions for campaigns and committee names are so ubiquitous, like Citizens For, Committee to Elect, Friends of, many unrelated campaigns get added together.

For future iterations I could try calculating the average silouhette score for each cluster, and see if clusters with scores above a certain threshold could be used for string cleaning. 

In [50]:

from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

to_cat = spending_data[(spending_data['form_type'].isin(['SB22','SB23','SB29'])) & (spending_data['vendor'].isna())].payee_id.unique().tolist()

tfidf_vect = TfidfVectorizer(max_df=5000, stop_words='english')
tfidf = tfidf_vect.fit_transform(to_cat)
tfidf.shape

(35235, 19232)

In [51]:

hdb = HDBSCAN(max_cluster_size = 5)

labels = hdb.fit_predict(tfidf)


In [52]:
# create df to examine clusters
data_res = pd.DataFrame({'text': to_cat, 'cluster':labels})

# exclude unclustered data 
data_res[data_res['cluster']!=-1].sort_values(by = 'cluster')

,text,cluster
12477,HOULE LACEY,0
12476,TRUJILLO ALISSA,0
12475,BOURGEOIS CHLOE,0
12474,ABUHAIKAL BARAA,0
12480,HERDER SUSAN,0
12527,HIXSON JENNIFER,1
12528,HAEUSSLER DEREK,1
12529,JOHNSTON GUZMAN ALYSSA,1
12530,BEUKER KAITLYN,1
12537,SHEPPARD DESTINY,1


### Classic clustering method 2: Agglomerative Clustering

I also tried Hierarchical Clustering. My theory here was that this methodology would be well suited to string cleaning because you could try to group strings by looking at the impact of changing one additional letter on the clusters and choosing the level that gets you closest to what you want. 

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances

# Pre calculate pairwise distance using already built tfidf matrix
dist_matrix = pairwise_distances(tfidf, metric='cosine')

# Initiate Agglomerative Clustering using the precomputed metric 
hierac_clustering = AgglomerativeClustering(
    n_clusters=None 
    , metric='precomputed' 
    , linkage='average'
    , distance_threshold=0.4
)

# Fit model
hierac_labels = hierac_clustering.fit_predict(dist_matrix) 

# Create DF to view results
data_res_hier = pd.DataFrame({'text': to_cat, 'cluster':hierac_labels})
data_res_hier.sort_values(by = 'cluster')